[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/GarretOS/python-ai-foundations/blob/main/projects/python-learning-resource-scraper/python_learning_resource_scraper.ipynb)
# 🐍 Python Learning Resource Scraper

This notebook teaches and demonstrates a small Python learning-resource scraper inspired by the Towards AI **Web Scraping** lesson.

## 🎯 Project Overview

The scraper fetches three predefined official Python documentation pages, parses their HTML, and reports each page's URL, title, first `<h1>` heading, and HTTP status. Threads fetch the pages concurrently, while indexed results preserve the original URL order.

This is a distinct portfolio application rather than a copy of the lesson example. It intentionally does not crawl the whole Python site or follow every link.

## 🐍 Python Concepts

This project practices:

- `requests` and HTTP response data
- BeautifulSoup HTML parsing
- Functions, parameters, arguments, and return values
- Default arguments with `verbose=False`
- A small lambda function
- LEGB scope with a global variable and nested helper function
- `threading.Thread`, `.start()`, and `.join()`
- Lists, dictionaries, loops, `enumerate()`, and f-strings
- `try` / `except requests.RequestException`
- The `if __name__ == "__main__":` execution guard

## 🌐 HTTP Requests with `requests`

The `requests` package sends an HTTP request and returns a response object. The scraper uses `response.text` as the HTML source and `response.status_code` as the result's status. A ten-second timeout keeps a slow request from waiting forever.

## 🍲 Parsing HTML with BeautifulSoup

`BeautifulSoup(response.text, "html.parser")` turns the returned HTML into an object that can be searched. The scraper checks `soup.title` for the page title and uses `soup.find("h1")` to get the first main heading. Missing elements receive clear fallback text.

## 🔧 Default Function Arguments

`scrape_resource(url, verbose=False)` has a default argument. A normal call can omit `verbose`, while the scraper can pass `True` when it wants small diagnostic messages.

## ⚡ Lambda Functions

The title extraction uses one small lambda. It returns the title text when a `<title>` element is available and otherwise returns `No Title Found`. The conditional expression keeps this focused example readable.

## 🔍 Python LEGB Scope

The nested `show_info()` function reads `local_info` from the enclosing `scrape_resource()` function and `GLOBAL_VERSION` from the global scope. This demonstrates the Enclosing and Global parts of Python's LEGB name lookup. It runs only when verbose mode is enabled.

## 🧵 Threading

`run_scraper()` creates one `threading.Thread` for each URL. Calling `.start()` begins each request, and `.join()` makes the program wait until every worker has finished before printing the final summary. Each worker writes to its own list index, so completion timing does not change the display order.

## 🛡️ Error Handling

Network problems such as timeouts and connection failures are handled with `except requests.RequestException`. The function returns a useful fallback dictionary, allowing the other threads and the final summary to continue. Missing `<title>` and `<h1>` elements are handled with fallback text too.

## 🚀 Run the Resource Scraper

Run the complete code cell below to fetch the three official documentation pages and print the final summary.

In [ ]:
import threading

import requests
from bs4 import BeautifulSoup


GLOBAL_VERSION = "Resource Scraper v1.0"


def scrape_resource(url, verbose=False):
    local_info = "Scraping Python learning resource"

    def show_info():
        print("Info:", local_info)
        print("Version:", GLOBAL_VERSION)

    if verbose:
        show_info()

    try:
        response = requests.get(url, timeout=10)
        soup = BeautifulSoup(response.text, "html.parser")

        extract_title = lambda soup_obj: (
            soup_obj.title.string if soup_obj.title and soup_obj.title.string
            else "No Title Found"
        )
        page_title = extract_title(soup).strip()

        heading_tag = soup.find("h1")
        main_heading = (
            heading_tag.get_text(strip=True)
            if heading_tag
            else "No H1 Heading Found"
        )

        return {
            "url": url,
            "title": page_title,
            "heading": main_heading,
            "status": response.status_code,
        }

    except requests.RequestException as error:
        if verbose:
            print(f"Request failed for {url}: {error}")

        return {
            "url": url,
            "title": "Request Failed",
            "heading": "Request Failed",
            "status": None,
        }


def update_results(index, url, results, verbose):
    results[index] = scrape_resource(url, verbose)


def run_scraper():
    urls = [
        "https://docs.python.org/3/tutorial/",
        "https://docs.python.org/3/library/",
        "https://docs.python.org/3/reference/introduction.html",
    ]
    results = [None, None, None]
    threads = []

    for i, link in enumerate(urls):
        thread = threading.Thread(
            target=update_results,
            args=(i, link, results, True),
        )
        threads.append(thread)

    for thread in threads:
        thread.start()

    for thread in threads:
        thread.join()

    print("\n=== Python Learning Resources ===")
    for index, result in enumerate(results, start=1):
        status = result["status"] if result["status"] is not None else "Unavailable"
        print(f"\n{index}. {result['title']}")
        print(f"   Heading: {result['heading']}")
        print(f"   Status: {status}")
        print(f"   URL: {result['url']}")


def main():
    print("=== Python Learning Resource Scraper ===")
    print("Fetching three official Python documentation pages...")
    run_scraper()


if __name__ == "__main__":
    main()

## 🧪 Try It Yourself

Run `scrape_resource()` with a controlled invalid URL and `verbose=True` to observe the request fallback. You can also test a small HTML string with BeautifulSoup to see the `No Title Found` and `No H1 Heading Found` messages when those elements are absent. Keep experiments small and avoid sending unnecessary requests to the documentation site.

## 📚 What I Learned

I practiced sending HTTP requests, parsing HTML, extracting elements, and storing related values in dictionaries. I also connected default arguments, a lambda function, nested scope, exception handling, and basic threading in one focused application.

## 📝 Notes

- The scraper uses only three fixed public pages and does not recursively crawl links.
- Responsible scraping means respecting site rules and robots guidance and avoiding excessive requests.
- The project does not attempt to bypass restrictions or anti-bot systems.
- Network results and documentation titles may change over time.
- `threading` is part of Python's standard library and is not listed in `requirements.txt`.